In [ ]:
import ast
import math
from collections import Counter
from igraph import Graph
from itertools import combinations

#############################################################
# LOAD RULES
#############################################################

#############################################################
# JACCARD SIMILARITY
#############################################################

def jaccard_similarity(rule1, rule2):

    s1 = set(rule1)
    s2 = set(rule2)

    inter = len(s1.intersection(s2))
    union = len(s1.union(s2))

    if union == 0:
        return 0

    return inter / union

#############################################################
# COMMUNITY DETECTION
#############################################################

def detect_communities(graph):

    communities = graph.community_multilevel(
        weights=graph.es["weight"]
    )

    graph.vs["community"] = communities.membership

    return communities

#############################################################
# RULE GRAPH
#############################################################

def build_rule_graph(rules, threshold=0.5):

    g = Graph()

    g.add_vertices(len(rules))

    edges = []
    weights = []

    for i, j in combinations(range(len(rules)), 2):

        sim = jaccard_similarity(
            rules[i],
            rules[j]
        )

        if sim >= threshold:

            edges.append((i, j))
            weights.append(sim)

    g.add_edges(edges)

    g.es["weight"] = weights

    return g

def load_rules(file_name):

    rules = []

    with open(file_name, "r") as f:

        for line in f:

            line = line.strip()

            if line == "":
                continue

            rule = ast.literal_eval(line)

            attrs = []

            for att in rule[1]:

                attrs.append(
                    (att[0], str(att[1]))
                )

            rules.append(attrs)

    return rules


#############################################################
# WSC
#############################################################

def compute_wsc(rules):

    return sum(len(rule) for rule in rules)


#############################################################
# GLOBAL FREQUENCIES
#############################################################

def compute_global_frequencies(rules):

    counter = Counter()

    total_pairs = 0

    for rule in rules:

        for pair in rule:

            counter[pair] += 1
            total_pairs += 1

    frequencies = {}

    for pair, count in counter.items():

        frequencies[pair] = count / total_pairs

    return frequencies

#############################################################
# COMMUNITY REUSE FACTOR
#############################################################

def compute_reuse_factor(rules, membership):

    reuse = {}

    n_communities = max(membership) + 1

    for cid in range(n_communities):

        unique_pairs = set()

        total_pairs = 0

        for idx, rule in enumerate(rules):

            if membership[idx] != cid:
                continue

            total_pairs += len(rule)

            for pair in rule:
                unique_pairs.add(pair)

        if total_pairs == 0:

            reuse[cid] = 1.0

        else:

            reuse[cid] = len(unique_pairs) / total_pairs

    return reuse

#############################################################
# SHANNON RARITY
#############################################################

def rarity_rule(rule, frequencies):

    rarity = 0

    for pair in rule:

        rarity += frequencies[pair]

    return rarity / len(rule)


#############################################################
# NEW METRIC
#############################################################

def compute_new_metric(
        rules,
        membership,
        alpha=1.0,
        beta=1.0,
        gamma=1.0
):

    frequencies = compute_global_frequencies(
        rules
    )
    print("Frequency: ", frequencies
          1223123123)

    reuse_factor = compute_reuse_factor(
    rules,
    membership
)

    complexity = 0

    details = []

    for idx, rule in enumerate(rules):

        size = len(rule)

        rarity = rarity_rule(
            rule,
            frequencies
        )

        community = membership[idx]

        reuse = reuse_factor[community]
        print("Size: ", size ** alpha  , " Rarity: ", rarity,"Reuse: ", reuse)

        score = (
            (size ** alpha)
            *
            (0 + beta * rarity)
            *
            (0 + gamma * (reuse))
        )

        complexity += score

        details.append({

            "rule": idx,

            "size": size,

            "rarity": rarity,

            "complexity": score

        })

    return complexity, details


#############################################################
# PRINT COMPARISON
#############################################################

def compare_metrics(file_name):

    rules = load_rules(file_name)

    ####################################################
    # RULE GRAPH
    ####################################################

    graph = build_rule_graph(
        rules,
        threshold=0.5
    )

    ####################################################
    # COMMUNITY DETECTION
    ####################################################

    communities = detect_communities(graph)
    print("\nRule Graph")
    print("---------------------------")
    print("Nodes:", graph.vcount())
    print("Edges:", graph.ecount())
    print("Density:", round(graph.density(),4))

    print("\nCommunities:", len(communities))

    sizes = [len(c) for c in communities]

    print("Community sizes:", sizes)

    wsc = compute_wsc(rules)

    new_metric, details = compute_new_metric(
        rules,
        communities.membership,
        alpha=1,
        beta=1,
        gamma=1
    )

    print("=" * 60)

    print("Policy")

    print("=" * 60)

    print("Rules:", len(rules))

    print("WSC:", round(wsc, 2))

    print("New Metric:", round(new_metric, 2))

    print()

    print("=" * 60)

    print("Per-rule analysis")

    print("=" * 60)



#############################################################

if __name__ == "__main__":

    #compare_metrics("/Users/ddiaz/Documents/code/phd-thesis-lab/17_NuevoAnalisisP1/reglas_abac.txt")
    compare_metrics("/home/daniel/Documents/phd/phd-thesis-lab/17_NuevoAnalisisP1/reglas_abac.txt")


Rule Graph
---------------------------
Nodes: 75
Edges: 27
Density: 0.0097

Communities: 56
Community sizes: [3, 2, 6, 4, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 3, 1, 1, 2, 3, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Frequency:  {('RID', '10283'): 0.003125, ('ROLE_ROLLUP_1', '117961'): 0.1, ('ROLE_ROLLUP_2', '118343'): 0.01875, ('ROLE_FAMILY', '118424'): 0.015625, ('RID', '9743'): 0.003125, ('ROLE_FAMILY', '290919'): 0.021875, ('RID', '10212'): 0.003125, ('ROLE_ROLLUP_2', '118300'): 0.009375, ('RID', '10890'): 0.00625, ('RID', '15127'): 0.00625, ('ROLE_FAMILY_DESC', '290919'): 0.021875, ('RID', '9755'): 0.00625, ('RID', '14626'): 0.003125, ('ROLE_DEPTNAME', '117945'): 0.00625, ('ROLE_FAMILY', '292795'): 0.021875, ('ROLE_CODE', '117880'): 0.009375, ('ROLE_FAMILY', '19721'): 0.034375, ('ROLE_TITLE', '117879'): 0.009375, ('ROLE_ROLLUP_2', '118386'): 0.003125, ('ROLE_TITLE', '120344'): 0.003125, ('ROLE_DEPTNAME', '119598'): 0.00

El ejemplo es el siguiente: Me gustaria que la parte estructural (|r| ** alpha) quede igual. Pero la parte de la rareza de la regla me gustaria que el valor me quedara entre el 0 y el 1, siendo el 0 un valor muy frecuente y el 1 un poco frecuente. Recuerda que la frecuencia es con base en todos los atributo valor que aparecen en la política. COmo aquí voy a obtener frecuencias bajas, me gustaria que el 0 sería la frecuencia minima y el 1 la frecuencia máxima, y así mapaear todas las frecuencias para que queden en este rango. Para esto voy a promediar la rareza de cada par que aparece en la regla entera. Después en la parte de la detección de comunidades, igual, me gustaria obtener un valor entre el 0 y el 1 para cada regla. DOnde 0 es un valor muy frecuente y el 1 es un valor menos frecuente. Aquí puedes aplicar la misma situación de mapear los valores minimos para el 0 y el valor máximo para el 1. También boy a promediar. Entonces la complejidad para cada regla quedaria de la siguiente manera: r_i ** \alpha * (\beta * rareza(r_i)) * (reuso(r_i)*\sigma). Cómo ves esta idea.

Es interesante lo que proponer, pero mi objetivo principal es obtener un valor menor a WSC, si le pongo 1+ voy a tener la posibilidad de tener valores mayores a WSC si tengo un \beta=1. Para evitar eso colocaba la multiplicación simple. Como los valores quedan ya mapeados, es seguro que tanto rareza como resuo siempre será mayor que 0. Así, siempre obtenfre un valor menor a WSC normal. No lo crees?

In [ ]:
Es interesante lo que proponer, pero mi objetivo principal es obtener un valor menor a WSC, si le pongo 1+ voy a tener la posibilidad de tener valores mayores a WSC si tengo un \beta=1. Para evitar eso colocaba la multiplicación simple. Como los valores quedan ya mapeados, es seguro que tanto rareza como resuo siempre será mayor que 0. Así 
